In [1]:
import pandas as pd

sessions_data = pd.read_csv("sessions_data.csv")
web_data = pd.read_csv("cleaned_web_data.csv")
web_data[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
1,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
4,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
5,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test
6,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
9,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test


In [2]:
# We check the data type of date_time & correct it
web_data["date_time"].dtype

<StringDtype(na_value=nan)>

In [3]:
web_data["date_time"] = pd.to_datetime(web_data["date_time"])
web_data["date_time"].dtype

dtype('<M8[us]')

In [4]:
# We calculate the duration of every step
# First, we group the records by client and visit with the earliest timestamp first

timed_steps = (
    web_data
    .sort_values(["client_id", "visit_id", "date_time"])
    .reset_index(drop=True)
)

timed_steps[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
5,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test
6,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
9,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test


In [5]:
# We add a new column with the following timestamp so we can compare them after

timed_steps["next_step_time"] = (
    timed_steps.groupby(["client_id", "visit_id"])["date_time"].shift(-1)
)

timed_steps[["process_step", "date_time", "next_step_time", "variation"]].head(10)

,process_step,date_time,next_step_time,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,Test
4,confirm,2017-04-15 13:00:34,NaT,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,Test
9,confirm,2017-04-12 15:47:45,NaT,Test


In [6]:
# We calculate the time passed between steps and add a new column to display it

timed_steps["step_duration"] = (
    timed_steps["next_step_time"] - timed_steps["date_time"]
).dt.total_seconds()

timed_steps[["process_step", "date_time", "next_step_time", "step_duration", "variation"]].head(10)

,process_step,date_time,next_step_time,step_duration,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
4,confirm,2017-04-15 13:00:34,NaT,NaN,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
9,confirm,2017-04-12 15:47:45,NaT,NaN,Test


In [7]:
### NEW CODE FROM HERE ###

# how many sessions ended before reaching the step "confirm"

timed_steps[
    (timed_steps["next_step_time"].isna()) &
    (timed_steps["process_step"] != "confirm")
]["visit_id"].nunique()

32508

In [8]:
#how many sessions ended in each step

timed_steps[
    (timed_steps["next_step_time"].isna()) &
    (timed_steps["process_step"] != "confirm")
].groupby("process_step")["visit_id"].nunique()

process_step
start     19092
step_1     6640
step_2     2786
step_3     4000
Name: visit_id, dtype: int64

In [9]:
# Add a column to mention a step when it is the last one 

timed_steps["last_step"] = timed_steps["process_step"].where(
    timed_steps["next_step_time"].isna()
)

# Adding the column to the sessions_data df only when the values are not missing
sessions_data = sessions_data.merge(
    timed_steps[timed_steps["last_step"].notna()][
        ["client_id", "visit_id", "last_step"]
    ],
    on=["client_id", "visit_id"],
    how="left",
    validate="one_to_one"
)

In [10]:
sessions_data.head(10)

,client_id,visit_id,step_count,last_step
0,555,637149525_38041617439_716659,5,confirm
1,647,40369564_40101682850_311847,5,confirm
2,934,7076463_57954418406_971348,4,start
3,1028,557292053_87239438319_391157,9,step_1
4,1104,543158812_46395476577_767725,1,start
5,1104,643221571_99977972121_69283,1,start
6,1186,507052512_11309370126_442139,1,start
7,1186,795373564_99931517312_810896,3,step_2
8,1195,393817425_39015278493_996341,5,confirm
9,1197,71862471_21202285428_848395,7,confirm


In [11]:
# checking for mistakes in the code

timed_steps[timed_steps["client_id"] == 934]

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step_time,step_duration,last_step
10,56,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:36:30,Test,2017-04-18 02:37:02,32.0,NaN
11,55,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02,Test,2017-04-18 02:38:24,82.0,NaN
12,54,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24,Test,2017-04-18 02:38:52,28.0,NaN
13,53,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:52,Test,NaT,NaN,start


In [12]:
#how many last_step in each step

sessions_data["last_step"].value_counts()

last_step
confirm    36927
start      19093
step_1      6640
step_3      4001
step_2      2786
Name: count, dtype: int64